# Task 2: Model-mix optimization

In this unit we will focus on how to formulate mixed-integer problems using the `Gurobipy` package. We will also provide a brief introduction to using `Gurobi` as a powerful solver and demonstrate how to create Python programs using MS Visual Studio Code. 

In [ ]:
# First you can check, whether the required packages are already installed; if not, you'll receive a warning
# Use the `%` symbol to run shell commands directly in the notebook.
%pip show gurobipy
%pip show matplotlib
%pip show numpy
%pip show pandas

In [ ]:
# Install required packages
# %pip install gurobipy
# %pip install matplotlib
# %pip install numpy
# %pip install pandas

Import everything:

In [ ]:
# importing the required packages
import gurobipy as gp # package as interface for Gurobi
from gurobipy import GRB # Gurobi environment
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

## Part 1: Optimize using Gurobipy

Gurobi is a commercial solver. You need to get a license to run it.

## Obtaining a Free Academic License for Gurobi in Jupyter Notebook

Gurobi, renowned for its optimization capabilities, offers a free named-user academic license for qualified academic users, enabling them to utilize its comprehensive optimization functionalities. This guide outlines the process of obtaining and utilizing the Gurobi academic license within a Jupyter notebook environment. Please make sure that you are connected to the university network or VPN.

For thorough instructions, please visit the official [Gurobi Academic License page](https://www.gurobi.com/features/academic-named-user-license/).

### Step 1: Create a Gurobi Account
1. Navigate to the [Gurobi website](https://www.gurobi.com/).
2. Click on “SIGN IN” (located at the top right) and create a new account using your academic email address.

### Step 2: Download, Install, and Activate License
Follow the [detailed instructions](https://www.gurobi.com/features/academic-named-user-license/) provided by Gurobi to download, install, and activate your academic license.

### Step 3: Install `gurobipy` in your Python Environment
Install the Gurobi Python API, `gurobipy`, using pip. Ensure that the pip command corresponds to your Jupyter environment:
```bash
pip install gurobipy


In [ ]:
from gurobipy import *

### Data

In [ ]:
# Import .csv file (WorkTasks) and rename columns and save into DataFrame
#df = ...
df = df.rename(columns={'No.': 'no', 'Due date': 'due_date', 'Type': 'type'})  # rename coloumns

df.head(10)

# Import .csv file, sort, and save into DataFrame

In [ ]:
# Defining sets and parameters
d_all = np.arange('2020-05-11', '2020-05-31', dtype='datetime64[D]')  # what is happening here?

T_plan = 7  # what is happening here?

print(d_all)

In [ ]:
# DataFrame of tasks for optimization; 1st day is May 11

df_optimize = df[df["due_date"] >= np.datetime64('2020-05-11') ]
df_optimize = df_optimize[df_optimize["due_date"] <= np.datetime64('2020-05-11') + T_plan - 1 ]
df_optimize = df_optimize.reset_index(drop=True) # Reset the index numbers of the new df

df_optimize # df_optimize now includes all orders between May 11-17; sort by order number

In [ ]:
# Create and sort array of unique product types
types = pd.unique(df_optimize.type) # what is happening here?
types = np.sort(types) # what is happening here?

num_types = len(types) # calculate number of product types
num_tasks = len(df_optimize) # calculate number of tasks
cap = 10 # initialize capacity

d_task = df_optimize["due_date"].values.astype('datetime64[D]')  # what is happening here?
n_task = df_optimize["no"].values  # what is happening here?

In [ ]:
 # what is happening here?
task_types = np.zeros((num_tasks, num_types), dtype=int)
#task_types = np.zeros((num_tasks, num_types), dtype=bool)

print(task_types)

In [ ]:
 # what is happening here?
for i in range(num_types):
    task_types[:, i] = (df_optimize['type'] == types[i])
        
print(task_types)

## Model

### Production planning<a class="anchor" id="PP"></a>

#### Indices: 
$\hspace{5mm} t \hspace{30mm}$   Index of period, $t=1, 2, ... T$

$\hspace{5mm} i \hspace{30mm}$   Index of tasks, $i=1, 2, ... I$

$\hspace{5mm} j \hspace{30mm}$   Index of tasks type, $j=1, 2, ... J$

#### Parameters: 
$\hspace{5mm} DueDate_i \hspace{16mm}$   Due date of task $i$

$\hspace{5mm} TaskTypes_{i,j} \hspace{11mm}$   $1$ if task $i$ is of type $j$, $0$ else

$\hspace{5mm} Capacity \hspace{17mm}$   Capacity [number of tasks / period]

#### Decision variables: 
$\hspace{5mm} X_{t,i} \hspace{25mm}$   Assignment of task $i$ to period $t$

$\hspace{5mm} DueDev_{i} \hspace{17mm}$   Lateness of task $i$ 

#### Objective function:
**Minimize lateness**: $ \hspace{15mm} \min \quad \displaystyle\sum_{i=1}^I DueDev_i $

#### Constraints:
1) **Earliness** $\hspace{29mm} \sum_{t=1}^T(DueDate_i-t)\cdot X_{t,i} \leq DueDev_i \hspace{29mm} \forall i = 1, 2, ..., I $

2) **Tardiness** $\hspace{28mm} \sum_{t=1}^T(t-DueDate_i)\cdot X_{t,i} \leq DueDev_i \hspace{29mm} \forall i = 1, 2, ..., I $

3) **Capacity** $\hspace{29mm} \sum_{i=1}^I X_{t,i} \leq cap \hspace{63mm} \forall t = 1, 2, ..., T $

4) **Schedule each tasks** $\hspace{14mm}  \sum_{t=1}^T X_{t,i} = 1 \hspace{66mm} \forall i = 1, 2, ..., I $

4) **Not more than one hard** $\hspace{8mm}  \sum_{i=1}^I X_{t,i} \cdot TaskType_{i,1}  \leq 1 \hspace{45mm} \forall t = 1, 2, ..., T $

5) **Non-negativity:** $ \hspace{20mm} DueDev_i \geq 0 \hspace{66mm}    \forall i = 1, 2, ..., I $

5) **Binary Variable:** $ \hspace{20mm} X_{t,i} \in (0, 1) \hspace{69mm}    \forall i = 1, 2, ..., I; t = 1, 2, ..., T $


* * * 

>We formulate a MILP (Mixed-Integer Linear Program) to minimize lateness (positive and negative violations) for the defined lenght of the plan of one week.

In [ ]:
# Defining a model
m = gp.Model("MILP") # initialize mathematical program with name MILP

# Defining the variables
X = m.addVars(T_plan, num_tasks, vtype=GRB.BINARY, name="X") # initialize binary assignment variables (task --> period)
DueDev = m.addVars(num_tasks, vtype=GRB.CONTINUOUS, lb=0.0, name="DueDev") # initialize lateness variable

# Defining the objective function
m.setObjective( sum(DueDev[i] for i in range(num_tasks)), GRB.MINIMIZE) # objective function: minimize total lateness

# what is happening here?
m.addConstrs(( DueDev[i] >= sum( (d_task[i] - d_all[t]).astype(int) * X[t,i] for t in range(T_plan)) for i in range(num_tasks) ), name="Earliness")
m.addConstrs(( DueDev[i] >= sum( (d_all[t] - d_task[i]).astype(int) * X[t,i] for t in range(T_plan)) for i in range(num_tasks) ), name="Tardiness")

# Capacity constraint: 
# add a new constraint called CapacityRestriction: in each period there should not be more assigned tasks than there is capacity 

# Schedule each task
# add a new constraint called TaskExactlyOnce: each task has to be assigend to exactly one period

# Schedule not more than 1 hard tasks (type 1) each period
m.addConstrs(( sum( X[t,i] * task_types[i,0] for i in range(num_tasks)) <= 1 for t in range(T_plan)), name="OnlyOneHard")

# Suppress console output
m.setParam('OutputFlag', 0)

# solve MILP and print our objective value
m.optimize()

## Results

In [ ]:
print("Objective value: ", m.objVal, " found after ", m.Runtime, " seconds. Relative gap is: ", m.MIPGap)

In [ ]:
# Set lenght of interval for the analysis
T_analyze = 20

# Write assignment of tasks to periods to an array, print result
X_results = np.empty(shape=(T_analyze,cap), dtype ='object')
for period in range(T_analyze): # for each period
    print('day', period, ':  ', end = '')
    i=0
    for task in range(num_tasks): # for each task
        if X[period, task].X > 0 : # check, if task is assigned to period
            X_results[period, i] = n_task[task] # write number of assigend tasks into X_results array of period
            print(n_task[task], '\t', end = '')
            i+=1
    print() # Line Break

# Print array of plan
print()
print(X_results)


### Plots

In [ ]:
import matplotlib.pyplot as plt

# Get number of days
num_days = X_results.shape[0]

# Create a figure and a set of subplots
fig, ax = plt.subplots()

# Iterate through each day
for day in range(num_days):
    # Filter out None values
    tasks = [task for task in X_results[day] if task is not None]
    
    # Plot each task as a separate bar, stacking them by adjusting the bottom position
    bottom = 0
    for task in tasks:
        ax.bar(day, 1, bottom=bottom, color='lightblue', edgecolor='black', linewidth=0.5)
        # Add text
        ax.text(day, bottom + 0.5, str(task), va='center', ha='center')
        bottom += 1

# Set x ticks and labels
ax.set_xticks(range(num_days))
ax.set_xticklabels([day+1 for day in range(num_days)])

# Set labels
ax.set_ylabel('Tasks')
ax.set_xlabel('Day')
ax.set_title('Tasks per Day')

# Show the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Get number of days
num_days = X_results.shape[0]

# what is happening here?
color_dict = {"Type 1": "mediumaquamarine", "Type 2": 'lightcyan', "Type 3": 'teal', "Type 4": 'skyblue', "Type 5": 'dodgerblue'}

# Create a figure and a set of subplots
fig, ax = plt.subplots()

# Iterate through each day
for day in range(num_days):
    # Filter out None values
    tasks = [task for task in X_results[day] if task is not None]
    
    # Plot each task as a separate bar, stacking them by adjusting the bottom position
    bottom = 0
    for task in tasks:
        # Get the type of the task from the DataFrame
        task_type = df[df.no == task].type.values[0]
        
        # Get the color for this task type
        task_color = color_dict[task_type]
        
        ax.bar(day, 1, bottom=bottom, color=task_color, edgecolor='black', linewidth=0.5)
        # Add text
        ax.text(day, bottom + 0.5, str(task), va='center', ha='center')
        bottom += 1

# Set x ticks and labels
ax.set_xticks(range(num_days))
ax.set_xticklabels([day+1 for day in range(num_days)])

# Create legend
patchList = []
for key in color_dict:
    data_key = patches.Patch(color=color_dict[key], label=key)
    patchList.append(data_key)
ax.legend(handles=patchList, bbox_to_anchor=(1, 1))

# Set labels
ax.set_ylabel('Tasks')
ax.set_title('Tasks per Day')

# Show the plot
plt.show()

### Some questions to work with the model
Analyze your results: how does the planning interval influence the results? Try out 20 and 10 periods. What do we learn from that?

`...`

What is happening, if we deactivate the OnlyOneHard constraint? 

`...`

What if we wanted to treat earliness and tardiness differently? How do we need to modify the model?

`...`

Optional: document the model by extending the following code

Mission 1 accomplished 🏅 * take a break 🥐🍦 

## Part 2: Create Python programs using MS Visual Studio Code

The Jupyter notebooks are a very nice way to visualize the development of model code and to mix textual explanations with Python code that can be executed. That's the reason, why they are used for this course. However, they have a disadvantage when it comes to working with large models and more complex code. When working on the models for your papers, you might want to use a Code-Editor like MS Visual Studio Code. This allows you to run your code more flexible, by line or multiple lines, not limited to the notebook cells. Also the code is executed faster and you can split large code to multiple files.

### Installing Visual Studio Code

Download the installer from [Visual Studio Code](https://code.visualstudio.com/) and follow the installation instructions for your operating system.

Install the Jupyter extension by Microsoft from the extensions view.

<img src="pics/jupyterext.png" alt="Python Logo" width="700">


Here you find a selection of keybord shortcuts: https://code.visualstudio.com/shortcuts/keyboard-shortcuts-windows.pdf. 


### Option 1: Working with Jupyter notebooks in MS Visual Studio Code

You can keep on working with Jupyter notebooks. The file handling and coding, however, will be much more convenient. 

Proceed as follows: 

1. Open the EXPLORER (Ctrl + Shift + E) and set the working directory (Arbeitsbereich) to your current folder; you might have to close the current folder before (press Ctrl+K and then F)
2. Navigate to your existing Jupyter notebook file in the EXPLORER
3. Open the file 

Notice: make sure that the correct kernel is active (upper right corner)


<img src="pics/Explorer.png" alt="Screenshot" width="600">

### Option 2: Direct coding in VS

As a lean alternative to using notebooks you can also code in Python directly. Give it a try by following the description. Make sure that you read all comments BEFORE you get started. 

Create a new Python (.py) file in VS Code and copy the code (= all code boxes) from this notebook into the file. 

Run parts of the code (by marking it and pressing ctrl+enter) and the entire model from that file. 

Notice: If you can't load the input data from the CSV file with ``CSV.read("data/WorkTasks_csv.csv", ...")``, it's probably because your current working directory in VS Code is not the folder that your code file is in. To set the working directory to the right path, open the folder containing you code files in VS Code first and then open the file from the "Explorer" Window in VS Code. If you are still using in the wrong directory, you can also do the following: 
1. Open the folder, that contains your code files in the VS Code Explorer. 
2. Open the file containing your code and run the first lines of the Code, so VS Code activates Python. 
4. Right click into the "Explorer" window, where you should see the files in the folder your Code is in and choose "Python: Change to This Directory". 

Now the directory is set to your folder. You can check your current working directory by calling ``pwd()`` in the Terminal.

### Working with separate files

Try also to split up your code in two files, for example "input_data.py" and "main_model.py". 
Your can import a file in another one using the following code:

In [ ]:
# import input_data
# This cell should not be executed in the Jupyter notebook

### Working with functions

In Python, functions can be created where variables are only locally stored, which can help manage RAM usage more efficiently. Try to write a function which encapsulates your model and can be called to run it:

In [ ]:
# Defining a function
def run_MILP():

    print("Model:")
    # put model here

# Calling a function
run_MILP()

Ready to take some real challenges? The first planning task is knocking at the door.